# MinIO / S3 Parquet access

This notebook demonstrates reading Parquet files stored in **MinIO** (S3-compatible storage) using credentials loaded from `.env.local`.

| # | Topic |
|---|---|
| 1 | Setup — load credentials, configure SSRF allowlist, test connectivity |
| 2 | `S3Catalog` — convenience class for S3/MinIO |
| 3 | `ParquetReader` with explicit `fs=` |
| 4 | `ConnectionCatalog` + `filesystem_profile` + `ParquetDataResource` |
| 5 | Column projection and filter pushdown on remote data |
| 6 | PyArrow integration with remote Parquet |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti_data").exists():
    parent = PROJECT_ROOT.parent.resolve()
    if (parent / "src" / "boti_data").exists():
        PROJECT_ROOT = parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import datetime as dt

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from boti_data import (
    ConnectionCatalog,
    ParquetSink,
    ParquetPipeline,
)
from boti_data.connection_catalog import S3Catalog
from boti_data.parquet import ParquetDataConfig, ParquetDataResource
from boti_data import ParquetReader


## 1. Setup — load credentials, configure SSRF allowlist, test connectivity

The ETL credentials live in `.env.local` (repo root, git-ignored). The MinIO endpoint is on a private network, so we add the host to the SSRF allowlist.

In [2]:
# Verify the .env.local file exists
env_file = PROJECT_ROOT / ".env.local"
print(f".env.local exists: {env_file.exists()}")
print(f".env.local path: {env_file}")


.env.local exists: True
.env.local path: /Users/lvalverdeb/TeamDev/repo-split/boti-data/.env.local


In [3]:
# Add the MinIO private IP to the SSRF allowlist so FilesystemConfig accepts it
import boti.core.filesystem as fsmod

MINIO_HOST = "10.211.55.36"
fsmod.ENDPOINT_ALLOWLIST.add(MINIO_HOST)
print(f"Allowlisted {MINIO_HOST}")
print(f"Allowlist now contains: {fsmod.ENDPOINT_ALLOWLIST}")


Allowlisted 10.211.55.36
Allowlist now contains: {'10.211.55.36'}


In [4]:
# Quick connectivity test — can we reach MinIO and list the bucket?
from boti.core import create_filesystem

try:
    config = fsmod.FilesystemConfig.from_env_prefix("ETL_", env_file=PROJECT_ROOT / ".env.local")
    fs = create_filesystem(config)
    items = fs.ls(config.fs_path)
    print(f"Connected to MinIO. Bucket '{config.fs_path}' contains {len(items)} top-level items.")
    for item in items:
        print(f"  {item}")
    CONNECTED = True
except Exception as exc:
    print(f"Could not connect to MinIO: {exc}")
    CONNECTED = False


Could not connect to MinIO: The specified bucket does not exist


## 2. `S3Catalog` — convenience class for S3/MinIO

`S3Catalog` is a lightweight helper that wraps `FilesystemConfig` + `FilesystemAdapter` for S3-compatible object stores. It is the simplest way to start working with MinIO data.

In [5]:
if CONNECTED:
    s3 = S3Catalog("ETL_", env_file=PROJECT_ROOT / ".env.local")
    print(f"Storage path: {s3.storage_path}")
    print(f"Filesystem: {type(s3.fs()).__name__}")
    print()

    # List top-level directories under the bucket
    top = s3.ls()
    print(f"Bucket root ({len(top)} items):")
    for item in top:
        print(f"  {item}")
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


In [6]:
if CONNECTED:
    # Read a parquet file directly via S3Catalog
    parquet_path = "bronze/logistics/mobile/biometrics/partition_date=2026-01-02/part.0.parquet"
    with s3.open(parquet_path) as f:
        table = pq.read_table(f)
    print(f"Read {table.num_rows} rows, {table.num_columns} columns directly from MinIO")
    print(table.schema)
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


## 3. `ParquetReader` with explicit `fs=`

`ParquetReader` accepts an `fs` keyword for a pre-configured filesystem instance. The `S3Catalog` provides the filesystem, and we pass it to the reader. This is the most flexible approach: all of the reader's features (date-range filtering, filter pushdown, column projection) work on remote data.

In [7]:
if CONNECTED:
    s3 = S3Catalog("ETL_", env_file=PROJECT_ROOT / ".env.local")
    fs = s3.fs()

    with ParquetReader({
        "parquet_storage_path": "dst-etl/bronze/logistics/mobile/biometrics",
    }, fs=fs) as reader:
        # Filter by associate_id
        df = reader.load(
            return_type="pandas",
            filters={"associate_id": 4715},
        )
        print(f"Filtered load (associate_id=4715): {len(df)} rows")
        print(df[["id", "associate_id", "service_result", "latitude", "longitude"]].head())
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


In [8]:
if CONNECTED:
    s3 = S3Catalog("ETL_", env_file=PROJECT_ROOT / ".env.local")
    fs = s3.fs()

    with ParquetReader({
        "parquet_storage_path": "dst-etl/bronze/logistics/mobile/biometrics",
        "parquet_start_date": dt.date(2026, 1, 2),
        "parquet_end_date": dt.date(2026, 1, 4),
    }, fs=fs) as reader:
        df = reader.load(return_type="pandas")
        print(f"Date-range load (Jan 2–4): {len(df)} rows, {len(df.columns)} columns")
        print(df[["id", "associate_id", "biometric_dt"]].head())
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


In [9]:
if CONNECTED:
    s3 = S3Catalog("ETL_", env_file=PROJECT_ROOT / ".env.local")
    fs = s3.fs()

    with ParquetReader({
        "parquet_storage_path": "dst-etl/bronze/logistics/mobile/biometrics",
    }, fs=fs) as reader:
        # Composite filter with pushdown
        df = reader.load(
            return_type="pandas",
            filters={
                "latitude__gte": -85.0,
                "service_result": "Exitosa",
            },
        )
        print(f"Composite filter: {len(df)} rows")
        print(df[["id", "latitude", "longitude", "service_result"]].head(3))
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


## 4. `ConnectionCatalog` + `filesystem_profile` + `ParquetDataResource`

`ConnectionCatalog` is the canonical registry for named connection profiles. By registering a filesystem profile, `ParquetDataResource` can resolve it automatically — no need to pass filesystem instances around.

In [10]:
if CONNECTED:
    catalog = ConnectionCatalog()
    catalog.load_filesystem("etl", "ETL_", env_file=PROJECT_ROOT / ".env.local")
    print("Registered filesystem profiles:", list(catalog._filesystem_configs.keys()))
    print()

    # Inspect the resolved config
    cfg = catalog.filesystem_config("etl")
    print(f"  fs_type: {cfg.fs_type}")
    print(f"  fs_path: {cfg.fs_path}")
    print(f"  fs_endpoint: {cfg.fs_endpoint}")
    print(f"  fs_region: {cfg.fs_region}")
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


In [11]:
if CONNECTED:
    config = ParquetDataConfig(
        parquet_storage_path="dst-etl/bronze/logistics/mobile/biometrics",
        filesystem_profile="etl",
    )
    with ParquetDataResource(config, catalog=catalog) as resource:
        df = resource.load_filtered({"associate_id": 4715}).compute()
        print(f"load_filtered via filesystem_profile: {len(df)} rows")
        print(df[["id", "associate_id", "service_result"]].head())
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


In [12]:
if CONNECTED:
    config = ParquetDataConfig(
        parquet_storage_path="dst-etl/bronze/logistics/mobile/biometrics",
        filesystem_profile="etl",
        parquet_start_date=dt.date(2026, 1, 2),
        parquet_end_date=dt.date(2026, 1, 4),
        partition_on=["partition_date"],
    )
    with ParquetDataResource(config, catalog=catalog) as resource:
        table = resource.load_arrow(columns=["id", "associate_id", "latitude", "longitude"])
        print(f"Arrow table: {table.num_rows} rows × {table.num_columns} cols")
        print(table.schema)
        print()
        print(table.to_pandas().head())
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


## 5. Column projection and filter pushdown on remote data

Column projection and filter pushdown work identically on remote Parquet. Only the required columns and row groups are fetched from S3, reducing I/O.

In [13]:
if CONNECTED:
    s3 = S3Catalog("ETL_", env_file=PROJECT_ROOT / ".env.local")
    fs = s3.fs()

    with ParquetReader({
        "parquet_storage_path": "dst-etl/bronze/logistics/products/tracking",
        "parquet_start_date": dt.date(2026, 1, 2),
        "parquet_end_date": dt.date(2026, 1, 3),
    }, fs=fs) as reader:
        # Project only a few columns
        df = reader.load(
            return_type="pandas",
            columns=["id", "tracking_dt", "user_login", "description"],
            filters={"user_login__startswith": "sqc"},  # dict-style filter
        )
        print(f"Projected + filtered load: {len(df)} rows, columns={list(df.columns)}")
        print(df.head())
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


## 6. PyArrow integration with remote Parquet

`ParquetDataResource.load_arrow()` and `load_filtered_arrow()` return native PyArrow tables, ideal for zero-copy columnar processing.

In [14]:
if CONNECTED:
    config = ParquetDataConfig(
        parquet_storage_path="dst-etl/bronze/logistics/products/tracking",
        filesystem_profile="etl",
        parquet_start_date=dt.date(2026, 1, 2),
        parquet_end_date=dt.date(2026, 1, 4),
        partition_on=["partition_date"],
    )
    with ParquetDataResource(config, catalog=catalog) as resource:
        # PyArrow list-style filters for predicate pushdown
        pa_filters = [("product_id", "in", [407581889, 407908555])]
        table = resource.load_arrow(filters=pa_filters, columns=["id", "tracking_dt", "product_id", "description"])
        print(f"PyArrow filtered load: {table.num_rows} rows, {table.num_columns} cols")
        print(table.to_pandas().head())
        print()
        # Dict-style filter via load_filtered_arrow
        table2 = resource.load_filtered_arrow({"product_id": 407973103})
        print(f"Dict-style filtered arrow: {table2.num_rows} rows")
        print(table2.to_pandas().head())
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


In [15]:
if CONNECTED:
    # Dask-backed (lazy) load from S3 via load_files
    config = ParquetDataConfig(
        parquet_storage_path="dst-etl/bronze/logistics/products/tracking",
        filesystem_profile="etl",
        parquet_start_date=dt.date(2026, 1, 2),
        parquet_end_date=dt.date(2026, 1, 5),
        partition_on=["partition_date"],
    )
    with ParquetDataResource(config, catalog=catalog) as resource:
        ddf = resource.load_files()
        print(f"Dask DataFrame: {type(ddf).__name__}, npartitions={ddf.npartitions}")
        print(f"Columns: {ddf.columns.tolist()}")
        # Trigger compute
        result = ddf.compute()
        print(f"Computed: {len(result)} rows")
        print(result.head())
else:
    print("Skipping — MinIO not available")


Skipping — MinIO not available


### Summary

- **`S3Catalog`** — simplest on-ramp for S3/MinIO access; wraps credentials, filesystem creation, and basic I/O.
- **`ParquetReader` with `fs=`** — full-featured reader (filters, projections, date-range) on a pre-configured filesystem.
- **`ConnectionCatalog` + `filesystem_profile`** — canonical approach; register a named profile once and pass it to any resource.
- **`ParquetDataResource`** — low-level resource with PyArrow and Dask methods, all work on remote Parquet.
- **Filter pushdown and column projection** are pushed to the S3/Parquet layer, minimizing data transfer.
- **SSRF protection** blocks private IPs by default; add trusted hosts to `boti.core.filesystem.ENDPOINT_ALLOWLIST`.